# STAGE 13B.2 — TEMPORAL RECORD RECOVERY AND MATCHED-GROUP RECONCILIATION

**Malaysia-only FYP | Diagnostic and recovery eligibility review (read-only)**

This notebook reproduces the exact Stage 05B-QC **1 case + 2 controls** group eligibility logic using the archived annotated temporal master. A complete *case row* does not imply a complete *matched group*. It preserves the original Stage 09/12 results and **does not create replacement training datasets** before identity, temporal and source checks have passed.

**Key source rules:** `rainfall_valid`; `soil_moisture_complete`; strict historical `landcover_valid_strict`; `waterway_available`; complete 1:2 group. Historical road distance is not required.

# PART A — PATHS AND INPUT VALIDATION
## 1. Libraries and project paths

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

ANNOTATED_PATH = (
    DEV_DIR / "03_derived_features"
    / "temporal_features_final"
    / "temporal_feature_samples_final_annotated.csv"
)

ORIGINAL_FULL_PATH = (
    DEV_DIR / "03_derived_features"
    / "temporal_features_final"
    / "temporal_model_full_enhanced.csv"
)

ORIGINAL_SPATIAL_PATH = (
    DEV_DIR / "04_model_ready"
    / "temporal_model_full_enhanced_spatial.csv"
)

RAW_TEMPORAL_PATH = (
    DEV_DIR / "03_derived_features"
    / "temporal_features"
    / "temporal_feature_samples.csv"
)

POSITIVE_META_PATH = (
    DEV_DIR / "02_cleaned_inventory"
    / "malaysia_positive_inventory_ml_eligible.geojson"
)

OUTPUT_DIR = (
    DEV_DIR / "05_model_results"
    / "stage13b2_temporal_reconciliation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Stage 05B annotated master": ANNOTATED_PATH,
    "Stage 05B full subset": ORIGINAL_FULL_PATH,
    "Stage 09 full spatial dataset": ORIGINAL_SPATIAL_PATH,
    "Stage 05B raw feature samples": RAW_TEMPORAL_PATH,
    "ML-eligible positive metadata": POSITIVE_META_PATH,
}.items():
    print(name, "| EXISTS:", path.exists(), "|", path)

if not ANNOTATED_PATH.exists():
    raise FileNotFoundError(
        "Stage 05B-QC annotated master is required: "
        + str(ANNOTATED_PATH)
    )

Stage 05B annotated master | EXISTS: True | C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_final_annotated.csv
Stage 05B full subset | EXISTS: True | C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_full_enhanced.csv
Stage 09 full spatial dataset | EXISTS: True | C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_full_enhanced_spatial.csv
Stage 05B raw feature samples | EXISTS: True | C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features\temporal_feature_samples.csv
ML-eligible positive metadata | EXISTS: True | C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson


## 2. Read original annotated master and require exact matching fields

In [2]:
df = pd.read_csv(ANNOTATED_PATH, low_memory=False)

REQUIRED = [
    "sample_id",
    "matched_positive_id",
    "temporal_role",
    "reference_date",
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
    "landcover_harmonised",
    "landcover_reliability",
    "distance_waterway_current_m",
]

missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Stage 05B annotated master missing: {missing}")

if df["sample_id"].isna().any() or df["sample_id"].duplicated().any():
    raise ValueError("Missing or duplicate sample_id values; do not silently deduplicate")

if df["matched_positive_id"].isna().any():
    raise ValueError("Missing matched_positive_id; cannot verify 1:2 matching")

print("Master rows:", len(df))
print("Cases:", df["temporal_role"].eq("case").sum())
print("Controls:", df["temporal_role"].eq("matched_control").sum())
print("Matched group IDs:", df["matched_positive_id"].nunique())

if not set(df["temporal_role"].dropna()).issubset({"case", "matched_control"}):
    raise ValueError("Unexpected temporal_role categories")

Master rows: 567
Cases: 189
Controls: 378
Matched group IDs: 189


# PART B — REPRODUCE STAGE 05B-QC ROW RULES
## 3. Recalculate rules from values and compare with original saved flags

In [3]:
RAIN_COLS = [
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
]
MOISTURE_COLS = [
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
]

for col in RAIN_COLS + MOISTURE_COLS + ["distance_waterway_current_m"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

computed = pd.DataFrame(index=df.index)
computed["rainfall_complete"] = df[RAIN_COLS].notna().all(axis=1)
computed["rainfall_logical"] = (
    df["rain_1d_mm"].le(df["rain_3d_mm"])
    & df["rain_3d_mm"].le(df["rain_7d_mm"])
    & df["rain_7d_mm"].le(df["rain_30d_mm"])
)
computed["rainfall_valid"] = (
    computed["rainfall_complete"] & computed["rainfall_logical"]
)
computed["soil_moisture_complete"] = df[MOISTURE_COLS].notna().all(axis=1)
computed["water_background_invalid"] = (
    df["temporal_role"].eq("matched_control")
    & df["landcover_harmonised"].eq("WATER")
)
computed["landcover_valid_strict"] = (
    df["landcover_harmonised"].notna()
    & df["landcover_reliability"].eq("HIGH")
    & ~computed["water_background_invalid"]
)
computed["waterway_available"] = df["distance_waterway_current_m"].notna()
computed["full_temporal_row_valid"] = (
    computed["rainfall_valid"]
    & computed["soil_moisture_complete"]
    & computed["landcover_valid_strict"]
    & computed["waterway_available"]
)

flag_rows = []
for name in computed.columns:
    if name in df.columns:
        saved = df[name].astype("string").str.lower().map({"true": True, "false": False})
        mismatches = (saved.notna() & (saved != computed[name])).fillna(False)
        flag_rows.append({
            "flag": name,
            "rows_compared": int(saved.notna().sum()),
            "mismatch_count": int(mismatches.sum()),
        })
    else:
        flag_rows.append({
            "flag": name,
            "rows_compared": 0,
            "mismatch_count": np.nan,
        })
    df[name + "_recomputed"] = computed[name].astype(bool)

flag_comparison = pd.DataFrame(flag_rows)
flag_comparison.to_csv(OUTPUT_DIR / "saved_vs_recomputed_qc_flags.csv", index=False)
print(flag_comparison.to_string(index=False))

if flag_comparison["mismatch_count"].fillna(0).gt(0).any():
    print("WARNING: Stage 05B saved flags differ from reproduced logic. Review before recovery.")

                    flag  rows_compared  mismatch_count
       rainfall_complete            567               0
        rainfall_logical            567               0
          rainfall_valid            567               0
  soil_moisture_complete            567               0
water_background_invalid            567               0
  landcover_valid_strict            567               0
      waterway_available            567               0
 full_temporal_row_valid            567               0


## 4. Group-level reproduction: one case, exactly two controls, all rows valid

In [4]:
RULES = {
    "rainfall": "rainfall_valid_recomputed",
    "rainfall_soilmoisture": "hydro_recomputed",
    "landcover": "landcover_valid_strict_recomputed",
    "full_enhanced": "full_temporal_row_valid_recomputed",
}

df["hydro_recomputed"] = (
    df["rainfall_valid_recomputed"]
    & df["soil_moisture_complete_recomputed"]
)

rows = []
for gid, group in df.groupby("matched_positive_id", dropna=False, sort=False):
    one_case = group["temporal_role"].eq("case").sum() == 1
    two_controls = group["temporal_role"].eq("matched_control").sum() == 2
    item = {
        "matched_positive_id": gid,
        "n_rows": len(group),
        "n_cases": int(group["temporal_role"].eq("case").sum()),
        "n_controls": int(group["temporal_role"].eq("matched_control").sum()),
        "structure_valid": bool(one_case and two_controls and len(group) == 3),
    }
    for rule_name, flag in RULES.items():
        item["case_valid_" + rule_name] = bool(
            group.loc[group["temporal_role"].eq("case"), flag].all()
        ) if one_case else False
        item["controls_valid_" + rule_name] = bool(
            group.loc[group["temporal_role"].eq("matched_control"), flag].all()
        ) if two_controls else False
        item["group_valid_" + rule_name] = bool(
            item["structure_valid"] and group[flag].fillna(False).all()
        )
    rows.append(item)

group_audit = pd.DataFrame(rows)
group_audit.to_csv(OUTPUT_DIR / "exact_group_eligibility_audit.csv", index=False)

for name in RULES:
    print(name, "groups:", int(group_audit["group_valid_" + name].sum()),
          "rows:", int(group_audit["group_valid_" + name].sum()) * 3)

rainfall groups: 189 rows: 567
rainfall_soilmoisture groups: 158 rows: 474
landcover groups: 148 rows: 444
full_enhanced groups: 121 rows: 363


# PART C — IDENTIFY REAL ROOT CAUSES
## 5. Compare the reproduced full subset with the original Stage 05B dataset

In [5]:
original_full = pd.read_csv(ORIGINAL_FULL_PATH, low_memory=False) if ORIGINAL_FULL_PATH.exists() else pd.DataFrame()
original_spatial = pd.read_csv(ORIGINAL_SPATIAL_PATH, low_memory=False) if ORIGINAL_SPATIAL_PATH.exists() else pd.DataFrame()

expected_group_ids = set(
    group_audit.loc[group_audit["group_valid_full_enhanced"], "matched_positive_id"].astype(str)
)

if not original_full.empty:
    if "matched_positive_id" not in original_full:
        raise ValueError("Original Stage 05B full dataset lacks matched_positive_id")
    actual_ids = set(original_full["matched_positive_id"].dropna().astype(str))
    print("Reproduced eligible groups:", len(expected_group_ids))
    print("Original full groups:", len(actual_ids))
    print("Eligible but absent from original:", len(expected_group_ids - actual_ids))
    print("Original but not eligible under reproduced rules:", len(actual_ids - expected_group_ids))
else:
    actual_ids = set()
    print("Original Stage 05B full subset missing; reproduction cannot be crosschecked")

compare = group_audit.copy()
compare["in_original_stage05b_full"] = compare["matched_positive_id"].astype(str).isin(actual_ids)
compare["recomputed_but_not_original"] = (
    compare["group_valid_full_enhanced"] & ~compare["in_original_stage05b_full"]
)
compare["original_but_recomputed_invalid"] = (
    ~compare["group_valid_full_enhanced"] & compare["in_original_stage05b_full"]
)
compare.to_csv(OUTPUT_DIR / "original_vs_recomputed_group_membership.csv", index=False)

Reproduced eligible groups: 121
Original full groups: 121
Eligible but absent from original: 0
Original but not eligible under reproduced rules: 0


## 6. Identify whether failures occurred in the case or the two controls

In [6]:
reasons = []
CHECKS = [
    ("rainfall", "rainfall_valid_recomputed"),
    ("soil_moisture", "soil_moisture_complete_recomputed"),
    ("strict_landcover", "landcover_valid_strict_recomputed"),
    ("waterway", "waterway_available_recomputed"),
]

for gid, group in df.groupby("matched_positive_id", dropna=False, sort=False):
    found = group_audit.loc[group_audit["matched_positive_id"].eq(gid)].iloc[0]
    for _, row in group.iterrows():
        failed = [name for name, col in CHECKS if not bool(row[col])]
        extra = []
        if not bool(row["rainfall_complete_recomputed"]):
            extra.append("missing_rainfall")
        elif not bool(row["rainfall_logical_recomputed"]):
            extra.append("rainfall_window_order_failure")
        if not bool(row["landcover_valid_strict_recomputed"]):
            if pd.isna(row["landcover_harmonised"]):
                extra.append("landcover_missing_or_unmapped")
            if row["landcover_reliability"] != "HIGH":
                extra.append("landcover_reliability_not_high")
            if bool(row["water_background_invalid_recomputed"]):
                extra.append("control_over_water")
        reasons.append({
            "matched_positive_id": gid,
            "sample_id": row["sample_id"],
            "temporal_role": row["temporal_role"],
            "case_row_complete": bool(found["case_valid_full_enhanced"]),
            "group_full_valid": bool(found["group_valid_full_enhanced"]),
            "failed_rules": "; ".join(failed),
            "specific_reasons": "; ".join(extra),
            "reference_date": row.get("reference_date"),
            "landcover_source": row.get("landcover_source"),
            "landcover_reliability": row.get("landcover_reliability"),
            "landcover_harmonised": row.get("landcover_harmonised"),
        })

row_reasons = pd.DataFrame(reasons)
row_reasons.to_csv(OUTPUT_DIR / "row_level_exclusion_reasons.csv", index=False)

excluded = compare.loc[~compare["group_valid_full_enhanced"]].copy()
summary_rows = []
for _, item in excluded.iterrows():
    subgroup = row_reasons.loc[
        row_reasons["matched_positive_id"].eq(item["matched_positive_id"])
    ]
    case = subgroup.loc[subgroup["temporal_role"].eq("case")]
    controls = subgroup.loc[subgroup["temporal_role"].eq("matched_control")]
    summary_rows.append({
        "matched_positive_id": item["matched_positive_id"],
        "structure_valid": item["structure_valid"],
        "case_all_rules_pass": item["case_valid_full_enhanced"],
        "both_controls_all_rules_pass": item["controls_valid_full_enhanced"],
        "case_failure_reasons": " | ".join(case["specific_reasons"].fillna("").tolist()),
        "case_failed_rules": " | ".join(case["failed_rules"].fillna("").tolist()),
        "control_failure_reasons": " | ".join(controls["specific_reasons"].fillna("").tolist()),
        "control_failed_rules": " | ".join(controls["failed_rules"].fillna("").tolist()),
        "recovery_category": (
            "REVIEW_GROUP_STRUCTURE" if not item["structure_valid"]
            else "CONTROL_ONLY_FAILURE" if item["case_valid_full_enhanced"] and not item["controls_valid_full_enhanced"]
            else "CASE_AND_OR_CONTROL_FAILURE"
        ),
    })

recovery = pd.DataFrame(summary_rows)
recovery.to_csv(OUTPUT_DIR / "excluded_group_root_causes.csv", index=False)
print(recovery["recovery_category"].value_counts(dropna=False))

recovery_category
CASE_AND_OR_CONTROL_FAILURE    35
CONTROL_ONLY_FAILURE           33
Name: count, dtype: int64


# PART D — TEMPORAL DATE AND MATCHING VERIFICATION
## 7. Preserve source event dates; never infer timing from row order

In [7]:
df["reference_date_parsed"] = pd.to_datetime(
    df["reference_date"], errors="coerce"
)

source_date_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ["date", "year", "timestamp", "window_end"])
]

candidate_date_columns = pd.DataFrame([
    {
        "column": c,
        "present_rows": int(df[c].notna().sum()),
        "dtype": str(df[c].dtype),
    }
    for c in source_date_cols
])
candidate_date_columns.to_csv(OUTPUT_DIR / "source_date_column_inventory.csv", index=False)

ref_audit = df[[
    "sample_id", "matched_positive_id", "temporal_role", "reference_date"
]].copy()
ref_audit["reference_date_parsed"] = df["reference_date_parsed"]

if "event_date_parsed" in df.columns:
    ref_audit["event_date_parsed"] = pd.to_datetime(
        df["event_date_parsed"], errors="coerce"
    )
else:
    ref_audit["event_date_parsed"] = pd.NaT

ref_audit["reference_date_missing"] = ref_audit["reference_date_parsed"].isna()
ref_audit["event_date_missing"] = ref_audit["event_date_parsed"].isna()
ref_audit["reference_event_date_mismatch"] = (
    ref_audit["reference_date_parsed"].notna()
    & ref_audit["event_date_parsed"].notna()
    & ref_audit["reference_date_parsed"].ne(ref_audit["event_date_parsed"])
)

ref_audit.to_csv(OUTPUT_DIR / "case_and_control_date_audit.csv", index=False)
print("Missing reference dates:", int(ref_audit["reference_date_missing"].sum()))
print("Case event dates present:", int(ref_audit.loc[ref_audit["temporal_role"].eq("case"), "event_date_parsed"].notna().sum()))
print("Different reference/event dates (when both known):", int(ref_audit["reference_event_date_mismatch"].sum()))
print("Note: raster observation timestamps are not necessarily available in this master.")

Missing reference dates: 0
Case event dates present: 189
Different reference/event dates (when both known): 0
Note: raster observation timestamps are not necessarily available in this master.


## 8. Verify dates within each matched group

In [8]:
date_group_rows = []
for gid, group in ref_audit.groupby("matched_positive_id", dropna=False):
    dates = group["reference_date_parsed"].dropna()
    case_dates = group.loc[group["temporal_role"].eq("case"), "reference_date_parsed"]
    date_group_rows.append({
        "matched_positive_id": gid,
        "all_reference_dates_present": bool(group["reference_date_parsed"].notna().all()),
        "reference_dates_identical_across_group": bool(
            len(dates) == len(group) and dates.nunique() == 1
        ),
        "case_event_date_present": bool(
            group.loc[group["temporal_role"].eq("case"), "event_date_parsed"].notna().all()
        ),
        "reference_date_matches_case_event_date": bool(
            case_dates.notna().all()
            and group.loc[group["temporal_role"].eq("case"), "event_date_parsed"].notna().all()
            and case_dates.reset_index(drop=True).equals(
                group.loc[group["temporal_role"].eq("case"), "event_date_parsed"].reset_index(drop=True)
            )
        ),
    })

date_groups = pd.DataFrame(date_group_rows)
date_groups.to_csv(OUTPUT_DIR / "matched_group_date_consistency.csv", index=False)
print(date_groups.drop(columns="matched_positive_id").sum().to_string())

all_reference_dates_present               189
reference_dates_identical_across_group    189
case_event_date_present                   189
reference_date_matches_case_event_date    189


# PART E — RECOVERY DECISION GATE
## 9. Build conservative candidate list without rewriting the training inputs

In [9]:
selection = compare.merge(
    date_groups,
    on="matched_positive_id",
    how="left",
    validate="one_to_one"
)

selection["candidate_status"] = np.select(
    [
        selection["group_valid_full_enhanced"] & selection["in_original_stage05b_full"],
        selection["recomputed_but_not_original"],
        ~selection["structure_valid"],
        selection["case_valid_full_enhanced"] & ~selection["controls_valid_full_enhanced"],
    ],
    [
        "ALREADY_INCLUDED",
        "ELIGIBLE_BY_REPRODUCED_FLAGS_REVIEW_ORIGINAL_MISMATCH",
        "INVALID_MATCHED_GROUP_STRUCTURE",
        "CASE_VALID_CONTROL_FAILURE",
    ],
    default="CASE_AND_OR_CONTROL_QC_FAILURE"
)

selection["safe_to_automatically_append"] = False
selection.to_csv(OUTPUT_DIR / "case_recovery_decision_matrix.csv", index=False)

print(selection["candidate_status"].value_counts(dropna=False).to_string())

flags_mismatch = int(flag_comparison["mismatch_count"].fillna(0).sum())
reconstruction_mismatch = int(
    compare["recomputed_but_not_original"].sum()
    + compare["original_but_recomputed_invalid"].sum()
)

checks = pd.DataFrame([
    {
        "check": "Recomputed QC flags equal saved flags",
        "status": "PASS" if flags_mismatch == 0 else "REVIEW",
        "finding": flags_mismatch,
    },
    {
        "check": "Original subset equals reproduced eligible groups",
        "status": "PASS" if ORIGINAL_FULL_PATH.exists() and reconstruction_mismatch == 0 else "REVIEW",
        "finding": reconstruction_mismatch,
    },
    {
        "check": "Original full spatial benchmark file exists",
        "status": "PASS" if ORIGINAL_SPATIAL_PATH.exists() else "REVIEW",
        "finding": str(ORIGINAL_SPATIAL_PATH.exists()),
    },
    {
        "check": "All group reference dates consistent",
        "status": "PASS" if date_groups["reference_dates_identical_across_group"].all() else "REVIEW",
        "finding": int((~date_groups["reference_dates_identical_across_group"]).sum()),
    },
    {
        "check": "Weather/raster source observation timestamps externally verified",
        "status": "PENDING_MANUAL_SOURCE_AUDIT",
        "finding": "No raw acquisition-timestamp proof in these flags",
    },
    {
        "check": "Revised matched controls individually verified",
        "status": "PENDING_IF_RECOVERY_PROPOSED",
        "finding": "No control replacements performed",
    },
    {
        "check": "Prior model results preserved",
        "status": "PASS",
        "finding": "This notebook only writes to its own audit folder",
    },
])
checks.to_csv(OUTPUT_DIR / "stage13b2_decision_gate.csv", index=False)
print(checks.to_string(index=False))

print("\nOUTPUT FOLDER:", OUTPUT_DIR)

candidate_status
ALREADY_INCLUDED                  121
CASE_AND_OR_CONTROL_QC_FAILURE     35
CASE_VALID_CONTROL_FAILURE         33
                                                           check                       status                                           finding
                           Recomputed QC flags equal saved flags                         PASS                                                 0
               Original subset equals reproduced eligible groups                         PASS                                                 0
                     Original full spatial benchmark file exists                         PASS                                              True
                            All group reference dates consistent                         PASS                                                 0
Weather/raster source observation timestamps externally verified  PENDING_MANUAL_SOURCE_AUDIT No raw acquisition-timestamp proof in these flags
     

## Interpretation and next action

1. A missing feature in a *case* is not the only cause of exclusion. Stage 05B-QC applies strict rules to **all three rows** in the group.
2. `landcover_reliability != HIGH` (including `LOW_PARTIAL_YEAR`) fails strict land-cover eligibility even when a class is present. A control over `WATER` also fails.
3. The 36 formerly described as “complete cases” cannot be automatically reintroduced; check `CASE_VALID_CONTROL_FAILURE` and row-level reasons.
4. Reproduced vs original discrepancies require manual review of source tables and the pipeline sequence.
5. Do not create additional controls, invent dates, fill missing soil moisture, or update Stage 12 models automatically.
6. If any groups are genuinely recovered, rebuild the matched temporal dataset and **repeat LR / RF / XGBoost with the same outer/inner spatial-validation design** before production retraining.

**Please send:** `stage13b2_decision_gate.csv`, `case_recovery_decision_matrix.csv`, `excluded_group_root_causes.csv`, `row_level_exclusion_reasons.csv`, `matched_group_date_consistency.csv`, `original_vs_recomputed_group_membership.csv`, and `saved_vs_recomputed_qc_flags.csv`.